In [ ]:
%load_ext autoreload
%autoreload 2

### Check current torch version

In [ ]:
import torch
print(torch.__version__)

In [ ]:
import torch
from torch.utils.data import DataLoader
from datasets.dataset import VOCDataset, VOC_CLASSES, get_transform, YoloV2GridTransform, Letterbox, detection_collate_fn
from utils.util import read_anchors, decode_predictions, apply_nms, plot_bounding_boxes, denormalize, encode_yolo_target, decode_batch_predictions, plot_anchors
from models.model_torch import YOLOv2, train_model, wrap_yolo_loss, save_checkpoint, load_checkpoint
import matplotlib.pyplot as plt
from PIL import Image


In [ ]:
# class_names = read_classes("data/coco_classes.txt")
class_names = VOC_CLASSES
print(class_names)
anchors = read_anchors("data/yolo_anchors.txt")
model_image_size = (416, 416) # Same as yolo_model input layer size
print(anchors)
num_classes = len(class_names)
num_anchors = len(anchors)

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
# data_transforms = v2.Compose([
#     # transforms.Resize((416,416)),
#     # transforms.RandomRotation(degrees=15),
#     # transforms.RandomHorizontalFlip(),
#     # transforms.RandomAutocontrast(0.1),
#     v2.ToImage(),                          # 1. Converts PIL Image to a Tensor image
#     v2.Resize((416,416)),
#     v2.ToDtype(torch.float32, scale=True), # 2. Converts to Float32 AND sca# les values to [0.0, 1.0]
#     # v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Always last
# ])

encoded_target = True
batch_size = 64
grid_shape=(13,13)
grid_width = 416/grid_shape[1]
grid_height = 416/grid_shape[0]


target_transform=None
if encoded_target:
    target_transform=YoloV2GridTransform(base_shape=(*grid_shape,num_anchors,5+num_classes), anchors=anchors,grid_width= grid_width, grid_height=grid_height)

train_dataset = VOCDataset(
    root="data/VOC2007",
    split_file=(
        "data/VOC2007/"
        "ImageSets/Main/train.txt"
    ),
    batch_size=batch_size,
    grid_shape=grid_shape,
    anchors=anchors,
    transform=get_transform(train=True),
    target_transform=target_transform,
)

val_dataset = VOCDataset(
    root="data/VOC2007",
    split_file=(
        "data/VOC2007/"
        "ImageSets/Main/val.txt"
    ),
    batch_size=batch_size,
    grid_shape=grid_shape,
    anchors=anchors,
    transform=get_transform(train=False),
    target_transform=target_transform,
)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(len(train_dataset), len(val_dataset))

### For explore the datasets shape

In [ ]:
obj_count = 0
no_obj_count = 0

for batch , (x, y) in enumerate(train_dataloader):
    print(len(y), y.shape)
    # for target in y:
    #     # print(y)
    #     obj_count += len(target["labels"])
    # total_gt = sum(
    #     boxes.shape[0]
    #     for boxes in gt_boxes
    # )
    objectness = y[...,4]
    print("no_obj",objectness[objectness == 0].numel())
    print("obj",(y[..., 4] == 1).sum().item())
    # boxes, scores, labels = decode_batch_predictions(y, anchors, conf_threshold=.3)
    # print("labels", len(labels))
    break
print(obj_count)


In [ ]:
image, target = train_dataset[4]
print(type(image), type(target))
# print(target.shape) # 13, 13, 5, 25
# print(target[..., 4]) # 13, 13, 5
# print(target[..., 4:5]) # 13, 13, 5, 1

# labels = target["labels"]
# boxes = target["boxes"]
# label_strings = [
#     f"{VOC_CLASSES[c]}-{i}"
#     for i, c in enumerate(labels)
# ]
# print("boxes", boxes.shape, boxes)
# plot_anchors(anchors)
# plot_bounding_boxes(image, boxes, label_strings, enable_grid=True)
# encoded_target = True
# if encoded_target:
#     target = encode_yolo_target(target, anchors, grid_width, grid_height, grid_shape, num_classes)
#     print(target.shape)

#     boxes, scores, labels = decode_predictions(target, anchors, conf_threshold=.3)
#     print(boxes)
#     label_strings = [
#         f"{VOC_CLASSES[c]}-{i}"
    #     for i, c in enumerate(labels)
    # ]
    # plot_bounding_boxes(image, boxes, label_strings, enable_grid=True)
# else:
    # labels = target["labels"]
    # boxes = target["boxes"]
# target = encode_yolo_target(target, torch.tensor(anchors, device=device), grid_width, grid_height, grid_shape, num_classes)
decoded_box, scores, labels = decode_predictions(target, anchors, conf_threshold=.3)

print(decoded_box.shape)
print(decoded_box)
label_strings = [
    f"{VOC_CLASSES[c]}-{i}"
    for i, c in enumerate(labels)
]

plot_bounding_boxes(image, decoded_box, label_strings, enable_grid=False)

### Define yolov2 model

In [ ]:
learning_rate = 1e-3
weight_decay = 1e-4
epochs = 10 # use small epochs for transfer learning

model = YOLOv2(num_anchors, num_classes).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs, # number of epochs
    eta_min=1e-6
)
loss_fn = wrap_yolo_loss(loss_weight=[5, 1, 0.05, 1], anchors=anchors)


### Training and evaluation

In [ ]:
history = train_model(
    epochs, model, train_dataloader, val_dataloader, loss_fn, optimizer, scheduler,
    batch_size, num_classes, anchors, device,
    checkpoint_path="save/latest.pth",
    best_checkpoint_path="save/best.pth",
    resume=True,
)
print("train_model", history)


In [ ]:
import gc
import torch

del model
gc.collect()
torch.cuda.empty_cache()  # Clears un-used cached memory blocks

In [ ]:
checkpoint_path = "save/yolov2.pth"
model = YOLOv2(num_anchors, num_classes).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1, eta_min=1e-6)
checkpoint = load_checkpoint(checkpoint_path, model, optimizer, scheduler, map_location=device)
print(f"Checkpoint from epoch {checkpoint['epoch']} successfully loaded.")

In [ ]:
print(f"mAP:    {history['map_metric'][-1]['map']:.4f}")
print(f"mAP@50: {history['map_metric'][-1]['map_50']:.4f}")
print(f"mAP@75: {history['map_metric'][-1]['map_75']:.4f}")

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(15,5))
plt.plot(epochs, history["val_map"], label="Validation mAP")
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
checkpoint_path = "save/yolov2.pth"
save_checkpoint(
    checkpoint_path,
    epoch=len(history["train_loss"]) - 1,
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    metrics={"best_val_map": max(history["val_map"], default=0.0)},
)
print(f"Checkpoint successfully saved to {checkpoint_path}")

### Visualize prediction bounding box

In [ ]:
image, target = train_dataset[5]
boxes, scores, labels = decode_predictions(target, anchors)
print(boxes.shape, scores.shape, labels.shape)

# plt.imshow(image.permute(1, 2, 0).numpy())
# plt.axis("off")  
# plt.show()

label_strings = [
    f"{VOC_CLASSES[c]}-{i}"
    for i, c in enumerate(labels)
]

plot_bounding_boxes(image, boxes, label_strings, enable_grid=True)
# pred = model(image.view(1, *image.shape).to(device))
# print(pred[0].shape)

# boxes_pred, scores_pred, labels_pred = decode_predictions(pred[0], anchors, stride=32, conf_threshold=.5, iou_threshold=.6, enable_nms=False)
# print("decode_predictions", boxes_pred.shape, scores_pred.shape, labels_pred.shape)
# print("boxes_pred", boxes_pred[:10])
# print(scores_pred[:10])
# print(labels_pred[:10])
# label_strings = [
#     f"{VOC_CLASSES[c]}-{i}"
#     for i, c in enumerate(labels_pred)
# ]
# print(labels_pred)
# plot_bounding_boxes(image, boxes_pred, label_strings)